# Day 1.2 — Similarity & Naive Search

---

Yesterday we turned sentences into vectors. Today we **compare** them and build our very first search engine.

By the end of 75 minutes you'll be able to:

1. Explain **cosine similarity** in one sentence
2. Build a working "find the most similar document" function in pure Python + numpy
3. See exactly why this approach doesn't work when you have millions of documents


## 1. Cosine similarity in one sentence

**Cosine similarity = how close two arrows point in the same direction.**

That's it. No math today.

- Two arrows pointing the **same way** → similarity ≈ **1.0** (very similar text)
- Two arrows at **90°** → similarity ≈ **0.0** (unrelated text)
- Two arrows pointing **opposite ways** → similarity ≈ **-1.0** (in practice, rare with modern embeddings)

You'll see values like `0.87` or `0.42` in real code — think of them as "how close, from 0 to 1."

> **Aside:** there are other ways to measure similarity (dot product, Euclidean distance). In text search, **cosine is the default**. Use it and don't overthink it.


## 2. The library does the math for us

`sentence-transformers` ships a helper called `util.cos_sim`. You don't have to write the formula — you just call it.


In [ ]:
!pip install sentence-transformers --quiet

In [5]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

a = model.encode("A dog is running in the park")
b = model.encode("A puppy plays on the grass")
c = model.encode("The stock market crashed today")

print(f"dog vs puppy    : {util.cos_sim(a, b).item():.3f}")
print(f"dog vs stocks   : {util.cos_sim(a, c).item():.3f}")
print(f"puppy vs stocks : {util.cos_sim(b, c).item():.3f}")


dog vs puppy    : 0.366
dog vs stocks   : 0.016
puppy vs stocks : 0.040


**What you should see:**
- `dog vs puppy` scores high (~0.7) — they mean similar things.
- `dog vs stocks` scores low (~0.05) — unrelated topics.

That's semantic search in one line.


## 3. Building a mini search engine

A search engine is: given a **query**, find the top-k **most similar** documents. Let's build it.


In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# Our tiny "database" of documents
docs = [
    "Python is a popular programming language.",
    "The Eiffel Tower is located in Paris, France.",
    "Machine learning models are trained on data.",
    "Croissants are a famous French pastry.",
    "FastAPI is a modern Python web framework.",
    "Neural networks are inspired by the human brain.",
    "The Louvre museum houses the Mona Lisa.",
    "Django is another Python web framework.",
]

# Embed all docs ONCE (this is the slow step — do it in advance)
doc_vectors = model.encode(docs)

def search(query: str, top_k: int = 3):
    q_vec = model.encode(query)
    scores = util.cos_sim(q_vec, doc_vectors)[0]      # 1D tensor of scores
    top = np.argsort(-scores.numpy())[:top_k]         # indices of top-k
    return [(docs[i], float(scores[i])) for i in top]

for hit, score in search("what are the places i can visit in france?"):
    print(f"  {score:.3f}  {hit}")


  0.625  The Eiffel Tower is located in Paris, France.
  0.571  Croissants are a famous French pastry.
  0.565  The Louvre museum houses the Mona Lisa.


**Notice a few important things:**

- We embedded the documents **once** and reused the vectors. That's the whole point — embedding is slow, comparing is fast.
- The query is embedded fresh every time (it's just one short string, so it's cheap).
- We didn't ask for the word "web" — the search found "FastAPI" and "Django" anyway. **That's semantic search.**


## 4. Try a "trick" query

Let's search for something that doesn't share any words with the documents.


In [3]:
for hit, score in search("places to visit in France"):
    print(f"  {score:.3f}  {hit}")


  0.436  The Eiffel Tower is located in Paris, France.
  0.417  Croissants are a famous French pastry.
  0.329  The Louvre museum houses the Mona Lisa.


You should see the Eiffel Tower, Louvre, and croissant sentences at the top — even though none of them contain the word "visit" or "places." A pure keyword search would have failed completely.


## 5. So why do we need anything else?

Our search function does this on every query:

```
for each document in the database:
    compute similarity(query, document)
sort scores
return top-k
```

This is called **linear search** (or "brute force"). It's fine for 100 documents. For 100,000 documents it starts to lag. For 10 million? Painfully slow — you'd wait seconds per query.

Real search engines solve this by pushing vectors into a **vector database** — a purpose-built store that handles fast approximate nearest-neighbor lookup for you, plus persistence, metadata filtering, and multi-user access. That's Day 3.


## 6. Let's see the slowness for ourselves


In [4]:
import time
import numpy as np

# Simulate 50,000 fake documents by making random vectors
N = 50_000
dim = 384
fake_docs = np.random.randn(N, dim).astype("float32")
fake_query = np.random.randn(dim).astype("float32")

start = time.time()
# Naive: compute dot product with every document
scores = fake_docs @ fake_query
top10 = np.argsort(-scores)[:10]
elapsed = (time.time() - start) * 1000

print(f"Searched {N:,} docs in {elapsed:.1f} ms")
print(f"Top result index: {top10[0]}")


Searched 50,000 docs in 16.2 ms
Top result index: 45153


Try changing `N` to `500_000` — you'll feel the slowdown. Now imagine doing this for every user, every query, all day. That's why we need indexes.


## Recap

- **Cosine similarity** = how close two arrows point. `util.cos_sim` does the math for you.
- A **search engine** = embed docs once, embed the query fresh, sort by similarity, return top-k.
- Semantic search finds meaning even when the words are different.
- Naive linear search works up to a few thousand docs, then it gets slow.
- **Next class:** vector databases — ChromaDB, Pinecone, pgvector — the tools that make vector search fast, persistent, and multi-user.
